# Compression de modèles : distillation, élagage, factorisation, quantisation

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/cercle-des-experts/blob/main/bonnes-pratiques-ingenierie/compression-modeles-machine-learning-efficace/compression-modeles-machine-learning-efficace.ipynb)

📖 **Le mini-cours complet** : [Compression de modèles : un levier clé pour un ML efficace](https://www.monshotdata.com/p/compression-modeles-machine-learning-efficace)

🧭 **Le cours** : [Bonnes pratiques d'ingénierie › 1.3 Déploiement et optimisation de modèles](https://www.monshotdata.com/mini-cours/bonnes-pratiques-ingenierie#h2sc13)

---

Un modèle qui brille en laboratoire mais qui est trop lent ou trop gros pour tourner en production ne sert à rien. Ici tu vas appliquer **cinq techniques de compression** sur MNIST et mesurer à chaque fois ce que tu gagnes (taille, vitesse) et ce que tu perds (précision).

**À la fin de ce notebook, tu sauras :**

- comment un petit modèle **élève** apprend d'un gros modèle **professeur** (distillation) ;
- comment supprimer jusqu'à 60 % des poids ou des neurones d'un réseau sans perdre de précision (élagage) ;
- comment compresser une couche par factorisation de rang faible, et comment quantifier un modèle en entiers 8 bits.


### Avant de lancer

Un **CPU suffit** : l'ensemble tourne en quelques minutes. MNIST est téléchargé automatiquement dans `./data` par torchvision. Les chiffres (temps d'inférence surtout) dépendent de ta machine : compare les ordres de grandeur plutôt que les valeurs exactes.

**Quatre corrections par rapport à la version diffusée avec le mini-cours :** la perte affichée pour la distillation et pour le zero-pruning était mal calculée (mauvais dénominateur, accumulation hors de la boucle) ; elle n'avait aucun effet sur l'entraînement lui-même. Les deux cellules vides de fin ont aussi été supprimées. Enfin, la boucle de factorisation écrasait les poids du réseau (jusqu'au rang 1) avant la quantisation : elle travaille maintenant sur une copie, et la quantisation s'applique bien à un modèle entraîné.

🧪 *Testé le 20 novembre 2026 : exécution complète de haut en bas sur CPU (environ 7 minutes) avec Python 3.13, PyTorch 2.x, torchvision, torchao, SciPy et pandas. Les sorties affichées viennent de cette exécution ; les chiffres varient légèrement d'un lancement à l'autre.*

## 1. Mise en place

On importe PyTorch, torchvision (pour MNIST), pandas et tqdm. Les cinq techniques partagent les mêmes données.

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np
import pandas as pd

from time import time
from tqdm import tqdm
from torch.utils.data import DataLoader

### Les données : MNIST

60 000 images de chiffres manuscrits (28×28 pixels) pour l'entraînement, 10 000 pour le test. On les normalise entre -1 et 1.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

## 2. Technique n°1 : la distillation (Knowledge Distillation)

L'idée : entraîner un **gros modèle** (le professeur, précis mais lent), puis demander à un **petit modèle** (l'élève) de reproduire ses prédictions. L'élève apprend non seulement la bonne réponse, mais aussi la façon dont le professeur « hésite » entre les classes.

### 2.1 Le professeur (*Teacher*)

Un petit réseau convolutif (une couche de convolution, une couche cachée dense).

In [ ]:
class TeacherNet(nn.Module):
    def __init__(self):
        super(TeacherNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 5)
        self.pool = nn.MaxPool2d(5, 5)
        self.fc1 = nn.Linear(32 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x        

### 2.2 Fonction d'évaluation

La précision (*accuracy*) sur les 10 000 images de test.

In [ ]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            inputs, labels = data
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

### 2.3 Entraîner le professeur

In [ ]:
teacher_model = TeacherNet()
teacher_optimizer = optim.Adam(teacher_model.parameters(), lr=0.001)
teacher_criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(5):
    teacher_model.train()
    running_loss = 0.0
    
    for data in trainloader:
        inputs, labels = data
        teacher_optimizer.zero_grad()
        outputs = teacher_model(inputs)
        loss = teacher_criterion(outputs, labels)
        loss.backward()
        teacher_optimizer.step()
        
        running_loss += loss.item()
        
    teacher_accuracy = evaluate(teacher_model)
        
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(trainloader)}, Accuracy: {teacher_accuracy * 100:.2f}%")

Epoch 1, Loss: 0.2276287337711482, Accuracy: 97.70%
Epoch 2, Loss: 0.07198821151726392, Accuracy: 98.11%
Epoch 3, Loss: 0.05438451458568941, Accuracy: 98.43%
Epoch 4, Loss: 0.04416755209133319, Accuracy: 98.40%
Epoch 5, Loss: 0.03741971013009119, Accuracy: 98.76%


### 2.4 L'élève (*Student*)

Un réseau tout simple à deux couches denses, sans convolution : beaucoup moins coûteux à exécuter.

In [ ]:
class StudentNet(nn.Module):
    def __init__(self):
        super(StudentNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
student_model = StudentNet()
student_optimizer = optim.Adam(student_model.parameters(), lr=0.001)

### 2.5 La perte de distillation : la divergence de KL

Au lieu de comparer la sortie de l'élève aux vraies étiquettes, on mesure l'écart entre la distribution de probabilité de l'élève et celle du professeur avec la **divergence de Kullback-Leibler**. Plus elle est petite, plus l'élève imite le professeur.

In [ ]:
def knowledge_distillation_loss(student_logits, teacher_logits):
    p_teacher = F.softmax(teacher_logits , dim=1)
    p_student = F.log_softmax(student_logits, dim=1)
    loss = F.kl_div(p_student, p_teacher, reduction='batchmean')
    return loss

### 2.6 Entraîner l'élève

Le professeur reste figé (`.detach()`) : seul l'élève apprend.

In [ ]:
# Entraîner le modèle étudiant avec la distillation des connaissances
for epoch in range(5):  # Tu peux ajuster le nombre d'époques
    student_model.train()
    running_loss = 0.0
    
    for data in trainloader:
        inputs, labels = data
        student_optimizer.zero_grad()
        student_logits = student_model(inputs)
        teacher_logits = teacher_model(inputs).detach()  # Détacher la sortie du professeur pour éviter la rétropropagation
        loss = knowledge_distillation_loss(student_logits, teacher_logits)
        loss.backward()
        student_optimizer.step()
        
        running_loss += loss.item()
    
    student_accuracy = evaluate(student_model)
        
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(trainloader)}, Accuracy: {student_accuracy * 100:.2f}%")

Epoch 1, Loss: 0.34278711689306474, Accuracy: 93.70%
Epoch 2, Loss: 0.15991362114188704, Accuracy: 95.76%
Epoch 3, Loss: 0.10823645342840378, Accuracy: 96.35%
Epoch 4, Loss: 0.08677414818001646, Accuracy: 96.79%
Epoch 5, Loss: 0.07211354228237005, Accuracy: 97.02%


### 2.7 Comparer les temps d'inférence

L'élève est plus rapide que le professeur (de l'ordre de 30 % sur un CPU), pour une perte de précision de l'ordre de 2 points.

In [ ]:
%timeit evaluate(teacher_model)

1.14 s ± 59.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%timeit evaluate(student_model) # # le modèle student s'exécute plus rapidement

799 ms ± 37 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## 3. Technique n°2 : le zero-pruning (élagage par seuil)

On met à zéro tous les poids dont la valeur absolue est inférieure à un **seuil λ**. Un poids proche de zéro contribue très peu à la sortie : le supprimer change à peine les prédictions, et la matrice devient **creuse**, donc stockable de façon compacte.

### 3.1 Un réseau dense à 4 couches

In [ ]:
# Définir un réseau de neurones simple pour la classification MNIST
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            inputs, labels = data
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

### 3.2 L'entraîner

In [ ]:
net = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

for epoch in range(5):
    running_loss = 0.0
    net.train()
    for data in trainloader:
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs.view(-1, 28*28))
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    accuracy = evaluate(net)
        
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(trainloader)}, Accuracy: {accuracy * 100:.2f}%")

Epoch 1, Loss: 0.3222244466299505, Accuracy: 94.27%
Epoch 2, Loss: 0.14593354894765723, Accuracy: 96.61%
Epoch 3, Loss: 0.10941860456960892, Accuracy: 95.14%
Epoch 4, Loss: 0.08864574225620428, Accuracy: 96.13%
Epoch 5, Loss: 0.0776950548201162, Accuracy: 97.00%


In [ ]:
# Sauvegarder le modèle
torch.save(net, "net.pt")

### 3.3 Balayer plusieurs seuils

Pour chaque seuil, on élague, on compte les poids mis à zéro et on mesure la précision. Observe jusqu'où l'on peut aller avant que la précision ne s'effondre.

In [ ]:
thresholds = np.linspace(0, 0.1, 11) # plage de seuils
results = []
total_params = np.sum([param.numel() for name, param in net.named_parameters() if 'weight' in name])

for threshold in tqdm(thresholds):
    
    # appliquer l'élagage par zéro (zero-pruning)
    for name, param in net.named_parameters():
        if 'weight' in name: # Appliquer l'élagage aux paramètres de poids
            param.data[torch.abs(param.data) < threshold] = 0
            
    # Compter les paramètres nuls
    zero_params = np.sum([torch.sum(param == 0).item() for name, param in net.named_parameters() if 'weight' in name])
    
    accuracy = evaluate(net)
    
    results.append([threshold, accuracy, total_params, zero_params])

100%|██████████| 11/11 [00:10<00:00,  1.08it/s]


In [ ]:
results = pd.DataFrame(results, columns = ["Threshold", "Accuracy", "Original Params", "Zero Params"])
results["Zero percentage"] = 100*results["Zero Params"]/results["Original Params"]
results

,Threshold,Accuracy,Original Params,Zero Params,Zero percentage
0,0.00,0.9700,566528,0,0.000000
1,0.01,0.9698,566528,122114,21.554804
2,0.02,0.9714,566528,239999,42.363131
3,0.03,0.9726,566528,349815,61.747169
4,0.04,0.9627,566528,426926,75.358323
5,0.05,0.9241,566528,468957,82.777374
6,0.06,0.8801,566528,490821,86.636671
7,0.07,0.8593,566528,506675,89.435121
8,0.08,0.8548,566528,518959,91.603416
9,0.09,0.8530,566528,528604,93.305891


### 3.4 Choisir le seuil et appliquer l'élagage

Le seuil de 0,03 supprime environ 60 % des poids pour une précision quasi inchangée (à quelques dixièmes de point près, dans un sens ou dans l'autre : l'élagage joue aussi un rôle de régularisation).

In [ ]:
threshold = 0.03
net = torch.load('net.pt', weights_only=False)

# appliquer l'élagage par zéro (zero pruning)
for name, param in net.named_parameters():
    if 'weight' in name: # Appliquer l'élagage aux paramètres de poids
        param.data[torch.abs(param.data) < threshold] = 0

### 3.5 Gagner réellement de la place : les matrices creuses

Mettre des zéros ne réduit pas la mémoire utilisée : un tableau dense occupe la même taille qu'il soit plein de zéros ou non. Il faut passer à un format **creux** (CSR) qui ne stocke que les valeurs non nulles.

In [ ]:
import scipy.sparse as sp

sparse_weights = []

# Convertir les poids élagués en matrice creuse
for name, param in net.named_parameters():
    if 'weight' in name:
        
        np_weight = param.data.cpu().numpy()
        sparse_weights.append(sp.csr_matrix(np_weight))

Taille des poids **avant** (format dense) :

In [ ]:
total_size = 0

for name, param in net.named_parameters():
    if 'weight' in name:
        tensor = param.data
        total_size += tensor.element_size() * tensor.numel()
        
# Convertir les octets en un format plus lisible (par ex., mégaoctets)
tensor_size_mb = total_size/(1024**2)

tensor_size_mb

2.1611328125

Taille des poids **après** (format creux CSR) :

In [ ]:
total_size = 0

for w in sparse_weights:
    total_size += w.data.nbytes
    
csr_size_mb = total_size/(1024**2)
csr_size_mb

0.8266944885253906

## 4. Technique n°3 : l'élagage des activations

Au lieu de supprimer des poids isolés, on supprime des **neurones entiers** : ceux dont l'activation moyenne est faible n'apportent presque rien. Le réseau devient réellement plus petit (moins de lignes et de colonnes dans les matrices), donc moins coûteux. Sur un réseau aussi petit et un CPU, le gain de temps reste toutefois modeste : c'est surtout la taille qui diminue.

### 4.1 Le réseau

Il renvoie aussi les activations intermédiaires, utiles pour mesurer l'importance des neurones.

In [ ]:
# Définir un réseau de neurones enseignant simple avec 4 couches entièrement connectées
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x1 = torch.relu(self.fc1(x))
        x2 = torch.relu(self.fc2(x1))
        x3 = torch.relu(self.fc3(x2))
        x4 = self.fc4(x3)
        return x1, x2, x3, x4  # Retourner les activations intermédiaires des couches pour l'élagage des activations

In [ ]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            inputs, labels = data
            outputs = model(inputs)[-1] # utiliser le dernier élément renvoyé par la fonction forward
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

### 4.2 L'entraîner

In [ ]:
net = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

for epoch in range(5):
    net.train()
    running_loss = 0.0
    
    for data in trainloader:
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs[-1], labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    accuracy = evaluate(net)
        
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(trainloader)}, Accuracy: {accuracy * 100:.2f}%")

Epoch 1, Loss: 0.32451173857545484, Accuracy: 95.45%
Epoch 2, Loss: 0.14378580869213223, Accuracy: 96.07%
Epoch 3, Loss: 0.10941547380813928, Accuracy: 96.96%
Epoch 4, Loss: 0.08991654900122069, Accuracy: 96.17%
Epoch 5, Loss: 0.0721870395959343, Accuracy: 97.38%


### 4.3 Mesurer l'activation moyenne de chaque neurone

On fait passer tout le jeu d'entraînement dans le réseau et on moyenne les sorties de chaque neurone.

In [ ]:
net.eval()
all_activations = [torch.zeros(512), torch.zeros(256), torch.zeros(128)]
data_size = len(trainloader.dataset.targets)

with torch.no_grad():
    for data in trainloader:
        inputs, _ = data
        
        activations_fc1 = torch.relu(net.fc1(inputs.view(-1, 28*28)))
        activations_fc2 = torch.relu(net.fc2(activations_fc1))
        activations_fc3 = torch.relu(net.fc3(activations_fc2))
        
        all_activations[0] += torch.sum(activations_fc1, dim=0)
        all_activations[1] += torch.sum(activations_fc2, dim=0)
        all_activations[2] += torch.sum(activations_fc3, dim=0)
            
for idx, activations in enumerate(all_activations):
    all_activations[idx] = activations/data_size

### 4.4 Supprimer les neurones sous un seuil

Pour chaque seuil, on construit un réseau plus petit en gardant seulement les neurones assez actifs, puis on mesure précision, taille et temps d'inférence.

In [ ]:
thresholds = np.linspace(0, 1, 11)
results = []

original_total_params = sum(p.numel() for p in net.parameters())
for threshold in tqdm(thresholds):
    new_net = SimpleNet()
    
    new_net.fc1.weight = net.fc1.weight
    new_net.fc2.weight = net.fc2.weight
    new_net.fc3.weight = net.fc3.weight
    new_net.fc4.weight = net.fc4.weight
    
    new_net.fc1.bias = net.fc1.bias
    new_net.fc2.bias = net.fc2.bias
    new_net.fc3.bias = net.fc3.bias
    new_net.fc4.bias = net.fc4.bias
    
    new_net.fc1.weight = nn.Parameter(new_net.fc1.weight[all_activations[0]>=threshold])

    new_net.fc2.weight = nn.Parameter(new_net.fc2.weight[:, all_activations[0]>=threshold])
    new_net.fc2.weight = nn.Parameter(new_net.fc2.weight[all_activations[1]>=threshold])
    
    new_net.fc3.weight = nn.Parameter(new_net.fc3.weight[:, all_activations[1]>=threshold])
    new_net.fc3.weight = nn.Parameter(new_net.fc3.weight[all_activations[2]>=threshold])
    
    new_net.fc4.weight = nn.Parameter(new_net.fc4.weight[:, all_activations[2]>=threshold])
    
    
    new_net.fc1.bias = nn.Parameter(new_net.fc1.bias[all_activations[0]>=threshold])
    new_net.fc2.bias = nn.Parameter(new_net.fc2.bias[all_activations[1]>=threshold])
    new_net.fc3.bias = nn.Parameter(new_net.fc3.bias[all_activations[2]>=threshold])
    
    
    accuracies = 0
    total_time = 0
    for _ in range(7):
        
        start = time()
        accuracies += evaluate(new_net)    
        total_time += time() - start
        
    new_total_params = sum(p.numel() for p in new_net.parameters())

    results.append([threshold, 100*accuracies/7, original_total_params, new_total_params, total_time/7])

100%|██████████| 11/11 [01:04<00:00,  5.88s/it]


In [ ]:
results = pd.DataFrame(results, columns = ["Threshold", "Accuracy", "Original Params", "New Params", "Inference Time"])
results["Size Reduction"] = 1-results["New Params"]/results["Original Params"]
results

,Threshold,Accuracy,Original Params,New Params,Inference Time,Size Reduction
0,0.0,97.38,567434,567434,0.884523,0.000000
1,0.1,97.38,567434,193276,0.872077,0.659386
2,0.2,97.37,567434,182052,0.839537,0.679166
3,0.3,97.27,567434,169413,0.859793,0.701440
4,0.4,96.73,567434,151508,0.882597,0.732994
5,0.5,96.41,567434,135819,0.829778,0.760644
6,0.6,95.87,567434,125568,0.860905,0.778709
7,0.7,94.64,567434,118301,0.800813,0.791516
8,0.8,93.30,567434,107521,0.799094,0.810514
9,0.9,86.10,567434,94769,0.814855,0.832987


## 5. Technique n°4 : la factorisation de rang faible (*Low-rank Factorization*)

Une matrice de poids `W` (256×128) peut être approchée par le produit de deux matrices plus étroites grâce à la **décomposition en valeurs singulières (SVD)**. On ne garde que les `r` valeurs singulières les plus grandes : c'est le **rang** `r`. Plus `r` est petit, moins il y a de paramètres, au prix d'une approximation moins fidèle.

### 5.1 Le réseau

In [ ]:
# Définir un réseau de neurones enseignant simple avec 4 couches entièrement connectées
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x1 = torch.relu(self.fc1(x))
        x2 = torch.relu(self.fc2(x1))
        x3 = torch.relu(self.fc3(x2))
        x4 = self.fc4(x3)
        return x4 

In [ ]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            inputs, labels = data
            outputs = model(inputs) # utiliser le dernier élément renvoyé par la fonction forward
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

### 5.2 L'entraîner

In [ ]:
net = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

for epoch in range(5):
    net.train()
    running_loss = 0.0
    
    for data in trainloader:
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    accuracy = evaluate(net)
        
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(trainloader)}, Accuracy: {accuracy * 100:.2f}%")

Epoch 1, Loss: 0.3239492867896552, Accuracy: 93.65%
Epoch 2, Loss: 0.14390295262295585, Accuracy: 96.82%
Epoch 3, Loss: 0.1088221705342923, Accuracy: 96.99%
Epoch 4, Loss: 0.08823778527602157, Accuracy: 96.96%
Epoch 5, Loss: 0.07462508281541945, Accuracy: 96.59%


### 5.3 Combien d'opérations pour multiplier ces matrices ?

Multiplier trois matrices peut se faire dans des ordres différents, avec des coûts très différents. Cette fonction (programmation dynamique classique du *Matrix Chain Multiplication*) trouve l'ordre qui minimise le nombre d'opérations.

In [ ]:
def MatrixChainOrder(p, i, j):
 
    if i == j:
        return 0
 
    _min = sys.maxsize
     
    for k in range(i, j):
     
        count = (MatrixChainOrder(p, i, k)
             + MatrixChainOrder(p, k + 1, j)
                   + p[i-1] * p[k] * p[j])
 
        if count < _min:
            _min = count;
     
    return _min;

### 5.4 Essayer plusieurs rangs

On factorise la 3ᵉ couche pour chaque rang et on suit : précision, paramètres, opérations et temps. Regarde jusqu'à quel rang la précision reste stable. La boucle travaille sur une copie du réseau (`net_rang`) : `net` reste intact et sera réutilisé à la section 6.

In [ ]:
rank_values = [128, 100, 90, 80, 60, 50, 40, 30, 20, 10, 5, 2, 1]
results = []
original_total_params = 128*256
batch_size = 32

# On travaille sur une copie : `net` doit rester intact pour la quantisation (section 6)
import copy
net_rang = copy.deepcopy(net)

# factoriser les poids de la couche 3
U, S, V = torch.svd(net.fc3.weight)

for rank in tqdm(rank_values):

    # Tronquer U, S et V pour ne conserver que les 'rank' premiers composants
    U_low_rank = U[:, :rank]
    S_low_rank = torch.diag(S[:rank])
    V_low_rank = V[:, :rank]

    # Reconstruire la matrice de poids factorisée
    factorized_weight_matrix = torch.mm(U_low_rank, torch.mm(S_low_rank, V_low_rank.t()))

    # Remplacer la matrice de poids de la couche choisie par la matrice de poids factorisée
    net_rang.fc3.weight = nn.Parameter(factorized_weight_matrix)
    
    weight_list = [batch_size, 256, rank, rank, 128]
    
    if rank == 128:
        # utiliser la matrice de poids habituelle
        total_operations = batch_size*256*128
    else:
        # trouver le nombre minimal d'opérations
        total_operations = MatrixChainOrder(weight_list, 1, 4)

    accuracies = 0
    total_time = 0
    for _ in range(7):
        
        start = time()
        accuracies += evaluate(net_rang)
        total_time += time() - start
    
    # nombre total de paramètres des trois matrices
    new_total_params = 128*rank + rank**2 + rank*256

    results.append([rank, 100*accuracies/7, original_total_params, new_total_params, total_operations, total_time/7])

100%|██████████| 13/13 [01:24<00:00,  6.48s/it]


In [ ]:
results = pd.DataFrame(results, columns = ["Threshold", "Accuracy", "Original Params", "New Params", "Operations", "Inference Time"])
results

,Threshold,Accuracy,Original Params,New Params,Operations,Inference Time
0,128,96.59,32768,65536,1048576,0.894265
1,100,96.57,32768,48400,1548800,0.901301
2,90,96.54,32768,42660,1365120,0.911000
3,80,96.57,32768,37120,1187840,0.929642
4,60,96.58,32768,26640,852480,0.927540
5,50,96.57,32768,21700,694400,0.948317
6,40,96.67,32768,16960,542720,0.957931
7,30,96.59,32768,12420,397440,0.959495
8,20,96.71,32768,8080,258560,0.898570
9,10,96.80,32768,3940,126080,0.906834


## 6. Technique n°5 : la quantisation

On remplace les nombres à virgule flottante 32 bits par des **entiers 8 bits** : le modèle prend environ quatre fois moins de place et les calculs en entiers sont plus rapides sur la plupart des processeurs. Ici, la quantisation dynamique int8 de la bibliothèque `torchao`.

In [ ]:
# Quantifier le modèle en type de données int8
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig

# Créer une copie du modèle pour la quantification
import copy
quantized_model = copy.deepcopy(net)

# Appliquer la quantification dynamique int8
quantize_(quantized_model, Int8DynamicActivationInt8WeightConfig())

W1120 21:28:23.549000 727 dist-packages/torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Précision du modèle quantifié :

In [ ]:
evaluate(quantized_model)

0.9658

Précision du modèle d'origine, pour comparer :

In [ ]:
evaluate(net)

0.9659

## 7. Ce qu'il faut retenir

| Technique | Ce qu'elle supprime | Gain | Coût |
|---|---|---|---|
| Distillation | le modèle entier (on en entraîne un plus petit) | vitesse | ~2 points de précision |
| Zero-pruning | des poids isolés | mémoire (si format creux) | très faible jusqu'à ~60 % |
| Élagage d'activations | des neurones entiers | taille (la vitesse change peu sur un petit réseau) | faible jusqu'à un seuil |
| Rang faible | des paramètres redondants | paramètres, opérations | faible si le rang reste raisonnable |
| Quantisation | de la précision numérique | mémoire, vitesse | quasi nul ici |

Ces techniques se **combinent** : on peut distiller, élaguer puis quantifier le même modèle.

---

## Pour aller plus loin

- [KAN from scratch : implémentation complète en PyTorch](https://www.monshotdata.com/p/implementer-kan-from-scratch-pytorch) — une autre façon d'alléger un réseau : changer d'architecture
- [Apprentissage Fédéré : Entraîne l'IA, Sécurise les Données](https://www.monshotdata.com/p/guide-pratique-apprentissage-federe) — entraîner des modèles directement sur des appareils